## 바닐라 RNN(Vanilla RNN)
- 가장 단순한 형태의 RNN
- 시점 t에서의 은닉 상태는 시점 t-1에서의 은닉 상태와 시점 t에서의 입력값에 의해 계산
- 비교적 짧은 시퀀스에 대해서만 효과가 있음
- time step의 길이가 길수록 앞의 정보가 뒤로 충분히 전달 x
- 장기 의존성 문제(the problem of long-term dependencies)
    - ex. '나는 학생이다. (중략) 오늘은 월요일이고 따라서 나는 _ 에 가야한다.'
    - \_를 예측하기 위해서는 '나는 학생이다'라는 정보가 필요하지만 시점 t='\_'에서 시점 t='학생'의 정보가 충분히 전달되지 않을 수 있음
- <img src="img/zzh.png" width="400>
    - *편향 $b$는 생략*

## LSTM(장단기 메모리)
- 은닉층의 메모리 셀에 입력 게이트, 망각 게이트, 출력 게이트를 추가하여 장기 의존성 문제를 해결한 RNN의 한 종류
- **셀 상태(cell state)**: $C_t$로 표현, 시점 t에서의 셀 상태는 시점 t-1에서의 셀 상태와 시점 t에서의 입력값에 의해 계산
- <img src=img/zzi.png width=400>

### 입력 게이트, 망각 게이트, 출력 게이트
- 은닉 상태의 값과 셀 상태의 값을 구하기 위해 추가된 3개의 게이트
- 공통적으로 시그모이드 함수 $\sigma$를 사용

#### 1. 입력 게이트
- 현재 정보를 기억하기 위한 게이트
- <img src=img/zzj.png width=300>
- $i_t = \sigma(W_{x i}x_t + W_{h i}h_{t-1} + b_i)$
    - $W_{x i}$: 시점 t에서의 입력값 $x_t$에 대한 가중치 행렬
    - $W_{h i}$: 시점 t-1에서의 은닉 상태 $h_{t-1}$에 대한 가중치 행렬
    - $b_i$: 편향 벡터
- $g_t = \tanh(W_{x g}x_t + W_{h g}h_{t-1} + b_g)$
    - $W_{x g}$: 시점 t에서의 입력값 $x_t$에 대한 가중치 행렬
    - $W_{h g}$: 시점 t-1에서의 은닉 상태 $h_{t-1}$에 대한 가중치 행렬
    - $b_g$: 편향 벡터
- $0 $\le$ i_t $\le$ 1$ 와 $-1 $\le$ g_t $\le$ 1$를 가지고 선택된 기억할 정보의 양을 정함.

#### 2. 삭제 게이트
- 기억을 삭제하기 위한 게이트
- <img src=img/zzk.png width=300">
- $f_t = \sigma(W_{x f}x_t + W_{h f}h_{t-1} + b_f)$
    - $W_{x f}$: 시점 t에서의 입력값 $x_t$에 대한 가중치 행렬
    - $W_{h f}$: 시점 t-1에서의 은닉 상태 $h_{t-1}$에 대한 가중치 행렬
    - $b_f$: 편향 벡터
- $0 $\le$ f_t $\le$ 1$를 가지고 삭제할 정보의 양을 정함.

#### 셀 상태
- <img src="img/zzl.png" width="300">
- $C_t = f_t \odot C_{t-1} + i_t \odot g_t$
    - $C_{t-1}$: 시점 t-1에서의 셀 상태
    - $\odot$: 원소별 곱셈(element-wise multiplication)
    - $f_t$가 0이 된다면 $C_{t-1}$의 정보가 완전히 삭제. 즉, 과거 시점의 셀 상태(기억)을 완전히 소거 후 현재 시점에서 선택된 정보만 새로운 셀 상태로 저장하는 상태.

#### 3. 출력 게이트
- 기억한 정보를 출력하기 위한 게이트
- <img src="img/zzm.png" width="300">
- $o_t = \sigma(W_{x o}x_t + W_{h o}h_{t-1} + b_o)$
    - $W_{x o}$: 시점 t에서의 입력값 $x_t$에 대한 가중치 행렬
    - $W_{h o}$: 시점 t-1에서의 은닉 상태 $h_{t-1}$에 대한 가중치 행렬
    - $b_o$: 편향 벡터
- $0 < o_t < 1$를 가지고 출력할 정보의 양을 정함.

#### 은닉 상태
- $h_t = o_t \odot \tanh(C_t)$
    - $C_t$: 시점 t에서의 셀 상태
    - $\odot$: 원소별 곱셈(element-wise multiplication)

In [49]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import SimpleRNN
# 단어 벡터 차원: 5, 문장 길이: 4
train_X = [[0.1, 4.2, 1.5, 1.1, 2.8], [1.0, 3.1, 2.5, 0.7, 1.1], [0.3, 2.1, 1.5, 2.1, 0.1], [2.2, 1.4, 0.5, 0.9, 1.1]]
print(np.shape(train_X))
print()

# RNN은 3D 텐서 입력을 필요로 하므로 변환
train_X = [train_X]
train_X = np.array(train_X, dtype=np.float32)
print(train_X.shape)
print()


(4, 5)

(1, 4, 5)



In [57]:
rnn = SimpleRNN(3) # return_sequences=False, return_state=False
# False, False: 반환: 최종 은닉 상태 (N, 3) // N: 입력 데이터 배치 크기
hidden_state = rnn(train_X)

print("hidden:", hidden_state, "shape:", hidden_state.shape)

hidden: tf.Tensor([[ 0.36384863 -0.43990162 -0.46132755]], shape=(1, 3), dtype=float32) shape: (1, 3)


In [51]:
rnn1 = SimpleRNN(3, return_sequences=True)
# True, False: 모든 시점의 은닉 상태 (N, T, 3) // T: 시점
hidden_state1 = rnn1(train_X)

print("hidden:", hidden_state1, "shape:", hidden_state1.shape)

hidden: tf.Tensor(
[[[-0.9146804   0.98883647 -0.9992236 ]
  [-0.62659234  0.9435895  -0.9853028 ]
  [-0.89574134  0.96906954 -0.45657676]
  [-0.7164188   0.9899093  -0.7510615 ]]], shape=(1, 4, 3), dtype=float32) shape: (1, 4, 3)


In [ ]:
rnn2 = SimpleRNN(3, return_sequences=True, return_state=True)
# return_state=True: return_sequences 여부와 상관없이 마지막 시점의 은닉 상태 출력
# True, True: 모든 시점의 은닉 상태, 최종 은닉 상태 (N, T, 3), (N, 3)
hidden_state2, last_state2 = rnn2(train_X)

print("hidden:", hidden_state2, "shape:", hidden_state2.shape)
print("last hidden:", last_state2, "shape:", last_state2.shape)

hidden: tf.Tensor(
[[[-0.99794054 -0.7809098  -0.18895882]
  [-0.751473   -0.65687215  0.14284603]
  [-0.8475627  -0.78527564  0.8821357 ]
  [-0.6509917   0.27815053 -0.5051734 ]]], shape=(1, 4, 3), dtype=float32) shape: (1, 4, 3)
last hidden: tf.Tensor([[-0.6509917   0.27815053 -0.5051734 ]], shape=(1, 3), dtype=float32) shape: (1, 3)


In [56]:
rnn3 = SimpleRNN(3, return_sequences=False, return_state=True)
hidden_state3, last_state3 = rnn3(train_X)
# False, True: 최종 은닉 상태, 최종 은닉 상태 (N, 3), (N, 3)

print("hidden:", hidden_state3, "shape:", hidden_state3.shape)
print("last hidden:", last_state3, "shape:", last_state3.shape)

hidden: tf.Tensor([[-0.9484838  -0.31915504  0.635042  ]], shape=(1, 3), dtype=float32) shape: (1, 3)
last hidden: tf.Tensor([[-0.9484838  -0.31915504  0.635042  ]], shape=(1, 3), dtype=float32) shape: (1, 3)


In [20]:
from tensorflow.keras.layers import LSTM, Bidirectional

lstm = LSTM(3, return_sequences=False, return_state=True) # 셀 상태까지 반환
hidden_state4, last_state4, last_cell_state4 = lstm(train_X)

print("hidden:", hidden_state4, "shape:", hidden_state4.shape)
print("last hidden:", last_state4, "shape:", last_state4.shape)
print("last call:", last_cell_state4, "shape:", last_cell_state4.shape)


hidden: tf.Tensor([[0.04045728 0.23218977 0.11635685]], shape=(1, 3), dtype=float32) shape: (1, 3)
last hidden: tf.Tensor([[0.04045728 0.23218977 0.11635685]], shape=(1, 3), dtype=float32) shape: (1, 3)
last call: tf.Tensor([[0.25996095 0.88195276 0.50566375]], shape=(1, 3), dtype=float32) shape: (1, 3)


#### SimpleRNN과 LSTM의 반환값 비교

##### SimpleRNN(3, ...) 의 반환값
return_sequences|return_state|반환되는 아웃풋 구조|최종 반환 Tensor의 Shape
|---|---|---|---|
False (기본값)|False (기본값)|최종 은닉 상태 (1개)|(N, 3)
True|False|모든 시점의 은닉 상태 (1개)|(N, T, 3)
False|True|최종 은닉 상태, 최종 은닉 상태 (2개)|(N, 3), (N, 3)
True|True|모든 시점의 은닉 상태, 최종 은닉 상태 (2개)|(N, T, 3), (N, 3)

##### LSTM(3, ...) 의 반환값
|return_sequences|return_state|반환되는 아웃풋 구조 (개수)|반환 텐서들의 Shape
|---|---|---|---|
False (기본값)|False (기본값)|최종 은닉 상태 (1개)|(N, 3)
True|False|모든 시점의 은닉 상태 (1개)|(N, T, 3)
False|True|최종 은닉 상태, 최종 은닉 상태, 최종 셀 상태 (3개)|(N, 3), (N, 3), (N, 3)
True|True|모든 시점의 은닉 상태, 최종 은닉 상태, 최종 셀 상태 (3개)|(N, T, 3), (N, 3), (N, 3)

#### Bidirectional(LSTM(3, ...)) 의 반환값
return_sequences|return_state|반환되는 아웃풋 구조 (개수)|반환 텐서들의 Shape
|---|---|---|---|
False (기본값)|False (기본값)|최종 양방향 은닉 상태 (1개)|(N, 6) *(6 = 3 + 3)*
True|False|모든 시점의 양방향 은닉 상태 (1개)|(N, T, 6) *(6 = 3 + 3)*
False|True|최종 은닉, 정방향 h, 정방향 C, 역방향 h, 역방향 C (총 5개)|(N, 6), (N, 3), (N, 3), (N, 3), (N, 3)
True|True|모든 은닉, 정방향 h, 정방향 C, 역방향 h, 역방향 C (총 5개)|(N, T, 6), (N, 3), (N, 3), (N, 3), (N, 3)

Bidirectional에서 
1) return_sequences=False, return_state=True 일 때
    - 최종 은닉 상태는 정방향 LSTM의 마지막 시점의 은닉 상태와 역방향 LSTM의 첫번째 시점의 은닉 상태가 연결된 채 반환된다.(역방향 LSTM의 마지막 출력은 결국 첫번째 시점(t=1)의 은닉 상태이기 때문에 시간순 정렬하면 f(1, 2, 3, 4)와 b(1,2,3,4) 이렇게 연결된다는 뜻.)
    - 정방향 h와 역방향 h는 각각 정방향 LSTM의 마지막 시점의 은닉 상태와 역방향 LSTM의 첫번째 시점의 은닉 상태가 반환된다.

2) return_sequences=True, return_state=True 일 때
    - 정방향 h와 역방향 h는 각각 정방향 LSTM의 마지막 시점의 은닉 상태와 역방향 LSTM의 첫번째 시점의 은닉 상태가 반환된다.
        - ((t=1일때 정방향, 역방향의 상태), (t=2일때 정방향, 역방향의 상태), ...)
        - 역방향 계산의 스탭 순서는 4, 3, 2, 1 이지만 최종 반환은 1, 2, 3, 4로 정렬되어 반환됨
